# TSTL Phase R1 — Colab (GPU)

**Runtime → Restart session のあと、セル1から順に実行**（セル3だけ実行しない）。

**GPU 必須**: Runtime → Change runtime type → T4 GPU（`torch cuda: True` になること）

**時間**: セル3の `PRESET_NAME='quick'` 推奨。`full` は Full GRPO だけで約45分×25回≈十数時間。

In [1]:
# セル1: clone/update + sys.path + pip
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/blabo25226/NSNandTSTL.git"
BRANCH = "20260713_create_TSTL"
CLONE_DIR = Path("/content/NSNandTSTL")
ROOT = CLONE_DIR / "TSTL"
SRC = ROOT / "src"


def run(cmd, cwd=None):
    print("$", " ".join(cmd))
    subprocess.run(cmd, cwd=cwd, check=True)


def sync_repo() -> None:
    """Fetch + hard reset (avoids git pull merge errors on Colab)."""
    try:
        run(["git", "fetch", "origin", BRANCH], cwd=CLONE_DIR)
        run(["git", "checkout", BRANCH], cwd=CLONE_DIR)
        run(["git", "reset", "--hard", f"origin/{BRANCH}"], cwd=CLONE_DIR)
    except subprocess.CalledProcessError as exc:
        print(f"git sync failed ({exc}); re-cloning")
        shutil.rmtree(CLONE_DIR)
        run(["git", "clone", "--branch", BRANCH, "--depth", "1", REPO_URL, str(CLONE_DIR)])


def has_llm_freeze() -> bool:
    return (SRC / "llm_freeze.py").is_file()


if CLONE_DIR.is_dir() and not has_llm_freeze():
    print("Stale clone without llm_freeze — removing and re-cloning")
    shutil.rmtree(CLONE_DIR)

if not CLONE_DIR.is_dir():
    run(["git", "clone", "--branch", BRANCH, "--depth", "1", REPO_URL, str(CLONE_DIR)])
elif (CLONE_DIR / ".git").is_dir():
    sync_repo()

if not has_llm_freeze():
    raise FileNotFoundError(
        f"{SRC / 'llm_freeze.py'} missing. Push branch {BRANCH} to GitHub."
    )

src_str = str(SRC.resolve())
if src_str not in sys.path:
    sys.path.insert(0, src_str)
os.chdir(ROOT)

run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements-r.txt")])

# Drop cached llm_* modules so cell 3 picks up fresh code after git sync
for name in list(sys.modules):
    if name == "llm_grpo" or name.startswith("llm_"):
        del sys.modules[name]

print("ROOT:", ROOT)
print("llm_freeze:", has_llm_freeze())
print("sys.path[0]:", sys.path[0])

In [2]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

CKPT_ROOT = Path('/content/drive/MyDrive/TSTL/r1')
CKPT_ROOT.mkdir(parents=True, exist_ok=True)
print('checkpoint root:', CKPT_ROOT)

In [3]:
import sys
import importlib
import inspect
from pathlib import Path

SRC = Path("/content/NSNandTSTL/TSTL/src")
if not (SRC / "llm_freeze.py").is_file():
    raise RuntimeError("セル1を先に実行してください")
src_str = str(SRC.resolve())
if src_str not in sys.path:
    sys.path.insert(0, src_str)

for name in list(sys.modules):
    if name == "llm_grpo" or name.startswith("llm_"):
        del sys.modules[name]

import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

from llm_freeze import freeze_all_except_layers, num_transformer_layers
from llm_grpo import GrpoRunConfig, run_grpo_train
from llm_eval import accuracy_score
from llm_profile import default_r1_out_dir, profile_layers_from_scores, resume_layer_scan
from r1_presets import FULL, QUICK, STANDARD, estimate_grpo_forward_passes, layer_indices_to_scan

print('run_grpo_train:', inspect.signature(run_grpo_train))
print('GrpoRunConfig has tokenizer:', 'tokenizer' in GrpoRunConfig.__dataclass_fields__)

# quick | standard | full  — R1 では quick 推奨（full は十数時間級）
PRESET_NAME = 'quick'
PRESET = {'quick': QUICK, 'standard': STANDARD, 'full': FULL}[PRESET_NAME]

MODEL = 'Qwen/Qwen2.5-0.5B-Instruct'
TRAIN_N = PRESET.train_n
EVAL_N = PRESET.eval_n
STEPS = PRESET.steps
NUM_GENERATIONS = PRESET.num_generations
MAX_COMPLETION = PRESET.max_completion_length
LAYER_STRIDE = PRESET.layer_stride
LR = 1e-5
SEED = 42

def make_grpo_config(output_dir, layer_indices=None, *, save_model=True):
    return GrpoRunConfig(
        output_dir=output_dir,
        learning_rate=LR,
        num_train_steps=STEPS,
        num_generations=NUM_GENERATIONS,
        max_completion_length=MAX_COMPLETION,
        train_layer_indices=layer_indices,
        tokenizer=tokenizer if 'tokenizer' in globals() else None,
        save_model=save_model,
    )

print('preset:', PRESET)
print('torch cuda:', torch.cuda.is_available())
print('llm_freeze:', __import__('llm_freeze').__file__)

In [4]:
try:
    from llm_data import load_gsm8k_subset, to_grpo_rows
except ImportError:
    from datasets import load_dataset

    def load_gsm8k_subset(n_train, n_eval):
        ds = load_dataset('openai/gsm8k', 'main')
        train = ds['train'].select(range(n_train))
        test = ds['test'].select(range(n_eval))
        return train, test

    def to_grpo_rows(split):
        return [
            {'prompt': f"Question: {ex['question']}\nAnswer:", 'answer': ex['answer']}
            for ex in split
        ]

train_split, eval_split = load_gsm8k_subset(TRAIN_N, EVAL_N)
tokenizer = AutoTokenizer.from_pretrained(MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
grpo_train = to_grpo_rows(train_split)
print('train rows:', len(grpo_train))

In [5]:
def eval_model(model, split, tokenizer, max_new_tokens=64):
    preds, gold = [], []
    model.eval()
    for ex in split:
        prompt = f"Question: {ex['question']}\nAnswer:"
        inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
        text = tokenizer.decode(out[0], skip_special_tokens=True)
        preds.append(text)
        gold.append(ex['answer'])
    return accuracy_score(preds, gold)


base_model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.bfloat16, device_map='auto')
s_base = eval_model(base_model, eval_split, tokenizer)
print('S_base', s_base)

In [6]:
out_dir = CKPT_ROOT / 'scan_latest'
out_dir.mkdir(parents=True, exist_ok=True)
BASE_CKPT = out_dir / 'base_model'

full_dir = out_dir / 'full'
full_model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.bfloat16, device_map='auto')
full_model.save_pretrained(BASE_CKPT)
tokenizer.save_pretrained(BASE_CKPT)

run_grpo_train(full_model, grpo_train, make_grpo_config(full_dir, layer_indices=None, save_model=True))
s_full = eval_model(full_model, eval_split, tokenizer)
print('S_full', s_full)

In [ ]:
n_layers = num_transformer_layers(full_model)
scan_layers = layer_indices_to_scan(n_layers, LAYER_STRIDE)
print('num layers:', n_layers, '| scan:', scan_layers)
print('est. GRPO gen passes:', estimate_grpo_forward_passes(PRESET, len(scan_layers)))


def train_and_eval_layer(k: int) -> float:
    print(f'--- layer {k} ---')
    m = AutoModelForCausalLM.from_pretrained(BASE_CKPT, torch_dtype=torch.bfloat16, device_map='auto')
    run_grpo_train(
        m,
        grpo_train,
        make_grpo_config(out_dir / f'layer_{k}', layer_indices=[k], save_model=False),
    )
    return eval_model(m, eval_split, tokenizer)


result = resume_layer_scan(
    out_dir,
    n_layers,
    train_and_eval_layer,
    s_base=s_base,
    s_full=s_full,
    layer_indices=scan_layers,
    config={
        'model': MODEL,
        'preset': PRESET_NAME,
        'steps': STEPS,
        'lr': LR,
        'seed': SEED,
        'scan_layers': scan_layers,
    },
)
print(result.out_dir)
if result.contributions:
    print('best C', max(result.contributions.values()))